# Naive Bayes from Scratch

Build a complete spam classifier without using a ready-made machine-learning classifier. The goal is to understand every probability used by a Naive Bayes model.

## Learning objectives

1. Calculate the prior probability of each class.
2. Estimate feature likelihoods from training data.
3. Apply Laplace smoothing.
4. Use log-probabilities for numerically stable scoring.
5. Predict classes for unseen examples.
6. Calculate accuracy and a confusion matrix from scratch.
7. Explain one prediction using feature-level evidence.

## Dataset

Each row represents an email. The target class is either `ham` or `spam`. All input features are binary.

| Feature | Meaning |
|---|---|
| `contains_free` | The email contains the word *free*. |
| `contains_win` | The email contains the word *win*. |
| `contains_offer` | The email contains the word *offer*. |
| `contains_meeting` | The email contains the word *meeting*. |
| `has_link` | The email contains a hyperlink. |
| `many_caps` | The email contains an unusually large amount of uppercase text. |

The `split` column gives a fixed train/test split so that everyone evaluates the same examples.

In [11]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

# Create a toy dataset, balanced between labels
columns = [
    "contains_free",
    "contains_win",
    "contains_offer",
    "contains_meeting",
    "has_link",
    "many_caps",
    "label",
    "split",
]

rows = [
    [1, 1, 1, 0, 1, 1, "spam", "train"],
    [1, 0, 1, 0, 1, 0, "spam", "train"],
    [0, 1, 1, 0, 1, 1, "spam", "train"],
    [1, 1, 0, 0, 0, 1, "spam", "train"],
    [0, 1, 0, 0, 1, 0, "spam", "train"],
    [1, 0, 0, 0, 1, 1, "spam", "train"],
    [0, 0, 1, 0, 1, 1, "spam", "train"],
    [1, 1, 1, 0, 1, 1, "spam", "train"],
    [0, 0, 1, 0, 0, 0, "spam", "train"],
    [1, 0, 1, 0, 0, 0, "spam", "train"],
    [0, 1, 1, 0, 0, 1, "spam", "train"],
    [1, 1, 0, 0, 1, 0, "spam", "train"],
    [0, 0, 0, 1, 0, 0, "ham", "train"],
    [0, 0, 0, 1, 1, 0, "ham", "train"],
    [0, 0, 1, 1, 0, 0, "ham", "train"],
    [0, 0, 0, 0, 0, 0, "ham", "train"],
    [0, 0, 0, 1, 0, 1, "ham", "train"],
    [0, 0, 0, 0, 1, 0, "ham", "train"],
    [0, 1, 0, 1, 0, 0, "ham", "train"],
    [1, 0, 0, 1, 0, 0, "ham", "train"],
    [0, 0, 1, 0, 0, 0, "ham", "train"],
    [0, 0, 0, 1, 1, 0, "ham", "train"],
    [0, 0, 0, 0, 0, 1, "ham", "train"],
    [0, 0, 0, 1, 0, 0, "ham", "train"],
    [1, 1, 1, 0, 1, 1, "spam", "test"],
    [1, 0, 1, 0, 1, 0, "spam", "test"],
    [0, 1, 0, 0, 1, 1, "spam", "test"],
    [1, 1, 0, 0, 0, 0, "spam", "test"],
    [0, 0, 0, 1, 0, 0, "ham", "test"],
    [0, 0, 1, 1, 0, 0, "ham", "test"],
    [0, 0, 0, 0, 1, 0, "ham", "test"],
    [1, 0, 0, 1, 0, 0, "ham", "test"],
]

data = pd.DataFrame(rows, columns=columns)
data

,contains_free,contains_win,contains_offer,contains_meeting,has_link,many_caps,label,split
0,1,1,1,0,1,1,spam,train
1,1,0,1,0,1,0,spam,train
2,0,1,1,0,1,1,spam,train
3,1,1,0,0,0,1,spam,train
4,0,1,0,0,1,0,spam,train
5,1,0,0,0,1,1,spam,train
6,0,0,1,0,1,1,spam,train
7,1,1,1,0,1,1,spam,train
8,0,0,1,0,0,0,spam,train
9,1,0,1,0,0,0,spam,train


In [12]:
feature_columns = [
    "contains_free",
    "contains_win",
    "contains_offer",
    "contains_meeting",
    "has_link",
    "many_caps",
]

train_data = data[data["split"] == "train"].reset_index(drop=True)
test_data = data[data["split"] == "test"].reset_index(drop=True)

X_train = train_data[feature_columns]
y_train = train_data["label"]
X_test = test_data[feature_columns]
y_test = test_data["label"]

print("Training shape:", X_train.shape)
print("Test shape:    ", X_test.shape)
print("\nTraining class counts:")
print(y_train.value_counts())

Training shape: (24, 6)
Test shape:     (8, 6)

Training class counts:
label
spam    12
ham     12
Name: count, dtype: int64


## Task 1 — Calculate class priors

For each class $c$, calculate:

$$P(c) = \frac{N_c}{N}$$

Store the result in a Pandas `Series` named `priors`. Its index must contain the class labels.

In [21]:
classes = np.sort(y_train.unique())

# Calculate class counts and priors for each class
class_counts = y_train.value_counts()
priors = pd.Series(class_counts, index=classes) / len(y_train)

priors

,count
ham,0.5
spam,0.5


## Task 2 — Estimate likelihoods with Laplace smoothing

Treat each input column as a categorical feature. For feature $X_j$, observed value $v$, and class $c$, estimate:

$$P(X_j=v \mid C=c) = \frac{N_{j,v,c}+\alpha}{N_c+\alpha K_j}$$

where:

- $N_{j,v,c}$ is the number of training examples in class $c$ for which feature $X_j$ has value $v$.
- $N_c$ is the number of training examples in class $c$.
- $K_j$ is the number of possible values of feature $X_j$ observed in the training set.
- $\alpha$ is the smoothing parameter. Use $\alpha=1$.

Although this dataset uses 0 and 1, the implementation should estimate a separate probability for every observed feature value using the general categorical likelihood formula.

In [23]:
def estimate_likelihoods(X, y, class_labels, alpha=1.0):
    """Estimate P(feature=value | class) for categorical features.

    Parameters
    ----------
    X : pandas.DataFrame
        Categorical training features.
    y : pandas.Series
        Training labels aligned with X.
    class_labels : array-like
        Ordered class labels to estimate.
    alpha : float
        Laplace smoothing parameter.

    Returns
    -------
    pandas.DataFrame
        Rows are classes. MultiIndex columns contain (feature, value).
    """
    possible_values = {
        feature: np.sort(X[feature].unique()) for feature in X.columns
    }
    likelihood_columns = pd.MultiIndex.from_tuples(
        [
            (feature, value)
            for feature, values in possible_values.items()
            for value in values
        ],
        names=["feature", "value"],
    )
    likelihoods = pd.DataFrame(
        index=class_labels, columns=likelihood_columns, dtype=float
    )

    for class_label in class_labels:
        X_class = X[y == class_label]

        for feature, values in possible_values.items():
            number_of_values = len(values)

            for value in values:
                # Count the number of examples matching the current feature value in this class
                value_count = np.sum(X_class[feature] == value)

                # Apply Laplace smoothing to estimate the conditional probability
                likelihoods.loc[class_label, (feature, value)] = (
                    value_count + alpha
                ) / (len(X_class) + alpha * number_of_values)

    return likelihoods


alpha = 1.0
likelihoods = estimate_likelihoods(
    X_train, y_train, classes, alpha=alpha
)
likelihoods

feature contains_free           contains_win           contains_offer  \
value               0         1            0         1              0   
ham          0.857143  0.142857     0.857143  0.142857       0.785714   
spam         0.428571  0.571429     0.428571  0.571429       0.357143   

feature           contains_meeting            has_link           many_caps  \
value           1                0         1         0         1         0   
ham      0.214286         0.357143  0.642857  0.714286  0.285714  0.785714   
spam     0.642857         0.928571  0.071429  0.357143  0.642857  0.428571   

feature            
value           1  
ham      0.214286  
spam     0.571429

## Task 3 — Calculate log-scores

Naive Bayes assumes that features are conditionally independent given the class:

$$P(\mathbf{x} \mid C=c) = \prod_j P(X_j=x_j \mid C=c)$$

The unnormalized log-posterior score is:

$$\log score(c) = \log P(C=c) + \sum_j \log P(X_j=x_j \mid C=c)$$

Adding logarithms is more numerically stable than multiplying many small probabilities.

In [27]:
def calculate_log_scores(X, class_labels, class_priors, feature_likelihoods):
    """Calculate one Naive Bayes log-score per class and row.

    Parameters
    ----------
    X : pandas.DataFrame
        Categorical examples to score.
    class_labels : array-like
        Ordered class labels.
    class_priors : pandas.Series
        Prior probability of each class.
    feature_likelihoods : pandas.DataFrame
        P(feature=value | class) values with MultiIndex columns.

    Returns
    -------
    pandas.DataFrame
        Unnormalized log-posterior scores for all rows and classes.
    """
    scores = pd.DataFrame(index=X.index, columns=class_labels, dtype=float)

    for row_index, row in X.iterrows():
        for class_label in class_labels:
            # Start with the log-prior for the current class
            log_score = np.log(class_priors[class_label])

            for feature in X.columns:
                observed_value = row[feature]

                # Fetch conditional probability P(feature=value | class) and add its log score
                probability = feature_likelihoods.loc[
                    class_label, (feature, observed_value)]
                log_score = log_score + np.log(probability)

            scores.loc[row_index, class_label] = log_score

    return scores


test_log_scores = calculate_log_scores(
    X_test, classes, priors, likelihoods
)
test_log_scores

,ham,spam
0,-9.948240,-3.329768
1,-6.857197,-3.905132
2,-6.857197,-4.205237
3,-6.433383,-4.793023
4,-2.262078,-7.933337
5,-3.561361,-7.345550
6,-3.766155,-4.780601
7,-4.053837,-7.645655


## Task 4 — Predict the test classes

The predicted class is the class with the largest log-score:

$$\hat{y}=\arg\max_c \log score(c)$$

Normalization is not required because the denominator in Bayes' theorem is the same for every candidate class.

In [31]:
# Select the class with the highest log-score for each email
y_pred = test_log_scores.columns[np.argmax(test_log_scores.to_numpy(),
                                           axis=1)].to_numpy()

prediction_results = X_test.copy()
prediction_results["actual"] = y_test.to_numpy()
prediction_results["predicted"] = y_pred
prediction_results

,contains_free,contains_win,contains_offer,contains_meeting,has_link,many_caps,actual,predicted
0,1,1,1,0,1,1,spam,spam
1,1,0,1,0,1,0,spam,spam
2,0,1,0,0,1,1,spam,spam
3,1,1,0,0,0,0,spam,spam
4,0,0,0,1,0,0,ham,ham
5,0,0,1,1,0,0,ham,ham
6,0,0,0,0,1,0,ham,ham
7,1,0,0,1,0,0,ham,ham


In [32]:
assert len(y_pred) == len(y_test), "Create exactly one prediction per test example."
assert set(np.unique(y_pred)).issubset(set(classes)), "Predictions must be known classes."
print("Task 4 checks passed.")

Task 4 checks passed.


## Task 5 — Evaluate the classifier

Calculate accuracy from scratch:

$$Accuracy = \frac{\text{number of correct predictions}}{\text{number of test examples}}$$

Then construct a confusion matrix. Rows must represent **actual** classes and columns must represent **predicted** classes.

In [35]:
def confusion_matrix_from_scratch(y_true, y_predicted, class_labels):
    """Build a confusion matrix without a metrics library.

    Parameters
    ----------
    y_true : array-like
        Actual class labels.
    y_predicted : array-like
        Predicted class labels.
    class_labels : array-like
        Row and column order for the matrix.

    Returns
    -------
    pandas.DataFrame
        Confusion matrix with actual rows and predicted columns.
    """
    matrix = np.zeros((len(class_labels), len(class_labels)), dtype=int)
    label_to_position = {label: i for i, label in enumerate(class_labels)}

    for actual, predicted in zip(y_true, y_predicted):
        # Increment the corresponding cell [actual, predicted] in the confusion matrix
        matrix[label_to_position[actual], label_to_position[predicted]] += 1

    return pd.DataFrame(
        matrix,
        index=[f"actual_{label}" for label in class_labels],
        columns=[f"predicted_{label}" for label in class_labels],
    )


# Compute model accuracy using NumPy
accuracy = np.mean(y_pred == y_test)
confusion = confusion_matrix_from_scratch(y_test, y_pred, classes)

print(f"Accuracy: {accuracy:.3f}")
display(confusion)

Accuracy: 1.000


,predicted_ham,predicted_spam
actual_ham,4,0
actual_spam,0,4


## Task 6 — Explain one prediction

Select one test example and explain why the model predicted its class. Your explanation must mention:

1. The actual and predicted classes.
2. The log-score of each candidate class.
3. At least two features that support the predicted class.
4. At least one absent or present feature that weakens the alternative class.
5. Why the result is a model-based probability judgment rather than proof that the email is truly spam or ham.

In [37]:
sample_position = 0  # Can be changed to another position in the test set.
sample = X_test.iloc[sample_position]
actual_class = y_test.iloc[sample_position]
predicted_class = y_pred[sample_position]
sample_scores = test_log_scores.iloc[sample_position]

evidence_rows = []
for feature, observed_value in sample.items():
    probability = likelihoods.loc[
        predicted_class, (feature, observed_value)
    ]
    evidence_rows.append(
        {
            "feature": feature,
            "feature_value": observed_value,
            f"probability_used_given_{predicted_class}": probability,
            "log_contribution": np.log(probability),
        }
    )

feature_evidence = (
    pd.DataFrame(evidence_rows)
    .set_index("feature")
    .sort_values("log_contribution", ascending=False)
)

print("Actual class:   ", actual_class)
print("Predicted class:", predicted_class)
print("\nClass log-scores:")
display(sample_scores.to_frame("log_score"))
print("Feature evidence for the predicted class:")
display(feature_evidence)

Actual class:    spam
Predicted class: spam

Class log-scores:


,log_score
ham,-9.948240
spam,-3.329768


Feature evidence for the predicted class:


,feature_value,probability_used_given_spam,log_contribution
feature,,,
contains_meeting,0,0.928571,-0.074108
contains_offer,1,0.642857,-0.441833
has_link,1,0.642857,-0.441833
contains_free,1,0.571429,-0.559616
contains_win,1,0.571429,-0.559616
many_caps,1,0.571429,-0.559616
